# 카멜레온 RL 학습 코드 — 순차 해설 노트북

`src/` 의 학습 파이프라인(`train.py`, `network.py`, `buffer.py`, `ppo.py`)을
**`import` 하지 않고** 실행 순서대로 한 곳에 펼친 노트북.

흐름:
1. 설정(config) — Hydra `default.yaml` 을 dict 로
2. 네트워크 — PointNet + ActorCritic (`network.py`)
3. 롤아웃 버퍼 + GAE (`buffer.py`)
4. PPO 업데이트 (`ppo.py`)
5. 롤아웃 수집 — Unity 환경과 상호작용 (`train.py`)
6. 학습 루프 — 환경 연결 → 반복 학습

> 실제 실행하려면 Python 3.10 환경(`unity_rl_310`) + 빌드(`Builds/MainEnv/Chameleon_env.exe`) 필요.
> 셀을 위에서 아래로 정의만 해두고, 맨 아래 학습 루프에서 전부 사용한다.

## 0. 큰 그림

```
UnityEnvironment 연결
  └─ for iteration in range(max_iterations):
        collect_rollout()      # 환경을 굴려 buf_size 만큼 (관측,행동,보상) 수집
        buf.get(last_value)    # GAE 로 advantage / return 계산 + advantage 정규화
        ppo.update(batch)      # clip 목적함수로 정책/가치 n_epochs 업데이트
        log / save             # TensorBoard 기록, 체크포인트 저장
```

핵심 객체 3개: **ActorCritic**(정책+가치 신경망), **RolloutBuffer**(경험 저장+GAE), **PPO**(업데이트).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.distributions import Normal, Categorical
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.base_env import ActionTuple
from mlagents_envs.side_channel.engine_configuration_channel import EngineConfigurationChannel

## 1. 설정 (config)

원래는 Hydra 가 `config/default.yaml` 을 읽어 `cfg` 로 넘긴다. 여기서는 같은 값을 dict 로 직접 둔다.

In [ ]:
cfg = {
    # 환경
    "env_path": "Builds/MainEnv/Chameleon_env.exe",  # None 이면 에디터 Play 대기 모드
    "seed": 0,
    "time_scale": 20.0,     # 빌드에서 게임 시계 배속
    "no_graphics": True,    # 헤드리스 실행

    # 네트워크
    "pointnet_out": 128,

    # 학습 루프
    "max_iterations": 5000,
    "buf_size": 10240,      # iteration 당 수집 스텝 수
    "batch_size": 1024,
    "n_epochs": 3,          # rollout 1개당 PPO 업데이트 epoch
    "log_interval": 10,
    "save_interval": 200,
    "save_dir": "results/run_01",

    # PPO
    "lr": 3e-4,
    "clip_eps": 0.2,
    "vf_coef": 0.5,
    "ent_coef": 0.01,
    "max_grad_norm": 0.5,

    # GAE
    "gamma": 0.99,
    "lam": 0.95,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

## 2. 네트워크 (`network.py`)

관측이 두 갈래라 인코더도 두 개:
- **벡터 관측 7D**(자기 위치/속도/머리각/남은 모기 수) → MLP
- **모기 집합(가변 개수)** → PointNet

### 2.1 PointNetEncoder
가변 개수의 모기 포인트 `{(x,y,z,vx,vy,vz)}` 를 **고정 크기 벡터**로 만든다.
포인트마다 같은 MLP 를 통과시키고 **max pooling** → 개수가 몇이든 출력 크기 일정.
빈 슬롯(패딩)은 `-inf` 로 채워 pooling 에서 제외.

In [ ]:
class PointNetEncoder(nn.Module):
    def __init__(self, out_dim, in_dim=6):
        super().__init__()
        self.out_dim = out_dim
        # 포인트(모기 1마리)마다 적용되는 공유 MLP
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, out_dim),
        )

    def forward(self, points):
        # points: [B, N, 6]. 전부 0인 행 = 패딩(그 자리에 모기 없음)
        if points.shape[1] == 0:                       # BufferSensor 가 비었을 때
            return torch.zeros(points.shape[0], self.out_dim, device=points.device)

        mask = points.abs().sum(dim=-1) > 0            # [B, N] 실제 모기 위치 True
        x = self.mlp(points)                           # [B, N, out_dim]
        x = x.masked_fill(~mask.unsqueeze(-1), float("-inf"))  # 패딩 자리 제외용
        x = x.max(dim=1).values                        # max pooling -> [B, out_dim]
        x = torch.where(torch.isinf(x), torch.zeros_like(x), x)  # 모기 0마리면 0 벡터
        return x

### 2.2 ActorCritic
- 벡터 인코더 + PointNet 출력을 이어붙여(fusion) 공통 특징 `h` 생성
- **Actor**: 연속 행동(전후진·차체yaw·머리yaw·머리pitch)은 tanh-squashed Gaussian,
  이산 행동(대기/혀발사)은 Categorical
- **Critic**: `h` 로 상태가치 V(s)

메서드 3개:
- `get_action`: 행동 샘플 + log_prob + V (롤아웃 수집 때)
- `evaluate`: 버퍼에 저장된 행동의 현재 정책 log_prob/entropy/V (PPO 업데이트 때)
- `get_value`: V 만 (truncation bootstrap 용)

연속 행동은 `tanh` 로 [-1,1] 로 짜내며, 그 변환의 log-det-Jacobian 보정을 log_prob 에 더한다.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, vec_dim, pointnet_out, cont_dim, disc_sizes):
        super().__init__()
        # 벡터 관측 인코더
        self.vec_encoder = nn.Sequential(
            nn.Linear(vec_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
        )
        self.pointnet = PointNetEncoder(out_dim=pointnet_out)
        # 두 인코더 출력 결합
        self.fusion = nn.Sequential(nn.Linear(64 + pointnet_out, 128), nn.Tanh())

        # Actor: 연속(Gaussian 평균 + 학습가능 log_std), 이산(브랜치별 Categorical)
        self.cont_mean = nn.Linear(128, cont_dim)
        self.log_std = nn.Parameter(torch.zeros(cont_dim))
        self.disc_heads = nn.ModuleList([nn.Linear(128, s) for s in disc_sizes])
        # Critic
        self.value_head = nn.Linear(128, 1)

    def _encode(self, vec_obs, point_cloud):
        h_vec = self.vec_encoder(vec_obs)              # [B, 64]
        h_pt = self.pointnet(point_cloud)              # [B, pointnet_out]
        return self.fusion(torch.cat([h_vec, h_pt], dim=-1))  # [B, 128]

    def get_action(self, vec_obs, point_cloud):
        h = self._encode(vec_obs, point_cloud)

        # 연속: tanh-squashed Gaussian
        mean = self.cont_mean(h)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)
        u = dist.rsample()                             # 원시 가우시안 샘플
        cont = torch.tanh(u)                           # [-1,1] 로 squash
        log_prob = dist.log_prob(u).sum(-1) - torch.log(1.0 - cont.pow(2) + 1e-6).sum(-1)

        # 이산: 브랜치별 샘플, log_prob 누적
        disc_list = []
        for head in self.disc_heads:
            d = Categorical(logits=head(h))
            a = d.sample()
            log_prob = log_prob + d.log_prob(a)
            disc_list.append(a)

        value = self.value_head(h).squeeze(-1)
        return cont, disc_list, log_prob, value

    def evaluate(self, vec_obs, point_cloud, cont_actions, disc_actions):
        h = self._encode(vec_obs, point_cloud)

        mean = self.cont_mean(h)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)
        # 저장된 행동([-1,1])을 atanh 로 원시 u 복원 (수치안정 clamp)
        cont_clamped = cont_actions.clamp(-1.0 + 1e-6, 1.0 - 1e-6)
        u = torch.atanh(cont_clamped)
        log_prob = dist.log_prob(u).sum(-1) - torch.log(1.0 - cont_clamped.pow(2) + 1e-6).sum(-1)
        entropy = dist.entropy().sum(-1)

        for i, head in enumerate(self.disc_heads):
            d = Categorical(logits=head(h))
            log_prob = log_prob + d.log_prob(disc_actions[:, i])
            entropy = entropy + d.entropy()

        value = self.value_head(h).squeeze(-1)
        return log_prob, entropy, value

    def get_value(self, vec_obs, point_cloud):
        h = self._encode(vec_obs, point_cloud)
        return self.value_head(h).squeeze(-1)

## 3. 롤아웃 버퍼 + GAE (`buffer.py`)

한 iteration 동안 `(관측, 행동, log_prob, 보상, V, done, bootstrap)` 을 차곡차곡 쌓고,
끝나면 **GAE** 로 advantage 와 return 을 계산한다.

- `collate_point_clouds`: 스텝마다 모기 수가 달라서, 미니배치로 묶을 때 `[B, N_max, 6]` 으로 0 패딩
- `compute_gae`: 뒤에서 앞으로 누적. **경계(done)** 에서 체인을 끊되,
  시간초과(truncated)면 terminal V 로 bootstrap, 진짜 종료면 0
- `get`: advantage 를 평균0/표준편차1 로 **정규화**(PPO 안정화의 핵심)

In [ ]:
def collate_point_clouds(clouds, device):
    # clouds: 길이 B 리스트, 각 [N_i, 6] -> [B, N_max, 6] (0 패딩)
    max_n = max((c.shape[0] for c in clouds), default=0) if clouds else 0
    if max_n == 0:
        return torch.zeros(len(clouds), 0, clouds[0].shape[-1], device=device)
    feat = clouds[0].shape[-1]
    out = torch.zeros(len(clouds), max_n, feat, device=device)
    for i, c in enumerate(clouds):
        if c.shape[0] > 0:
            out[i, :c.shape[0]] = c.to(device)
    return out


class RolloutBuffer:
    def __init__(self, buf_size, vec_dim, cont_dim, n_disc, gamma, lam, device):
        self.buf_size = buf_size
        self.gamma = gamma
        self.lam = lam
        self.device = device

        self.vec_obs = torch.zeros(buf_size, vec_dim)
        self.point_clouds = []                 # [N_i, 6] 리스트
        self.cont_acts = torch.zeros(buf_size, cont_dim)
        self.disc_acts = torch.zeros(buf_size, n_disc, dtype=torch.long)
        self.log_probs = torch.zeros(buf_size)
        self.rewards = torch.zeros(buf_size)
        self.values = torch.zeros(buf_size)
        self.dones = torch.zeros(buf_size)     # 에피소드 경계 -> GAE 체인 리셋
        self.bootstraps = torch.zeros(buf_size)  # 경계 스텝의 V(s_next)
        self.ptr = 0

    def add(self, vec_obs, point_cloud, cont_act, disc_act, log_prob,
            reward, value, done, bootstrap_value=0.0):
        i = self.ptr
        self.vec_obs[i] = vec_obs.cpu()
        self.point_clouds.append(point_cloud.cpu())
        self.cont_acts[i] = cont_act.cpu()
        self.disc_acts[i] = disc_act.cpu()
        self.log_probs[i] = log_prob.cpu()
        self.rewards[i] = reward
        self.values[i] = value.cpu()
        self.dones[i] = float(done)
        self.bootstraps[i] = bootstrap_value
        self.ptr += 1

    def compute_gae(self, last_value):
        advantages = torch.zeros(self.ptr)
        last_gae = 0.0
        for t in reversed(range(self.ptr)):
            done = self.dones[t].item()
            if done:
                next_val = self.bootstraps[t].item()   # 종료=0, 시간초과=V(terminal)
            elif t == self.ptr - 1:
                next_val = last_value
            else:
                next_val = self.values[t + 1].item()
            delta = self.rewards[t] + self.gamma * next_val - self.values[t]
            last_gae = delta + self.gamma * self.lam * (1.0 - done) * last_gae
            advantages[t] = last_gae
        returns = advantages + self.values[:self.ptr]
        return advantages, returns

    def get(self, last_value):
        advantages, returns = self.compute_gae(last_value)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)  # 정규화
        pc = collate_point_clouds(self.point_clouds[:self.ptr], self.device)
        return {
            "vec_obs": self.vec_obs[:self.ptr].to(self.device),
            "point_cloud": pc,
            "cont_acts": self.cont_acts[:self.ptr].to(self.device),
            "disc_acts": self.disc_acts[:self.ptr].to(self.device),
            "log_probs": self.log_probs[:self.ptr].to(self.device),
            "advantages": advantages.to(self.device),
            "returns": returns.to(self.device),
        }

    def reset(self):
        self.ptr = 0
        self.point_clouds.clear()

## 4. PPO 업데이트 (`ppo.py`)

수집한 배치를 미니배치로 쪼개 `n_epochs` 번 반복하며 업데이트.

- **ratio** = exp(새 log_prob − 옛 log_prob)
- **clip 목적함수**: `min(ratio·A, clip(ratio,1±ε)·A)` 를 최대화(→ 음수로 최소화)
- **가치 손실**: (return − V)^2
- **엔트로피 보너스**: 탐색 장려(손실에서 빼줌)
- 전체 손실 = policy_loss + vf_coef·value_loss + ent_coef·(−entropy)

In [ ]:
class PPO:
    def __init__(self, model, lr, clip_eps, vf_coef, ent_coef,
                 max_grad_norm, n_epochs, batch_size):
        self.model = model
        self.clip_eps = clip_eps
        self.vf_coef = vf_coef
        self.ent_coef = ent_coef
        self.max_grad_norm = max_grad_norm
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    def update(self, batch):
        vec_obs = batch["vec_obs"]
        point_cloud = batch["point_cloud"]
        cont_acts = batch["cont_acts"]
        disc_acts = batch["disc_acts"]
        old_log_prob = batch["log_probs"]
        advantages = batch["advantages"]
        returns = batch["returns"]

        n = vec_obs.shape[0]
        idx = torch.arange(n, device=vec_obs.device)
        total_pl, total_vl, total_ent = 0.0, 0.0, 0.0

        for _ in range(self.n_epochs):
            perm = idx[torch.randperm(n)]
            for start in range(0, n, self.batch_size):
                mb = perm[start:start + self.batch_size]
                log_prob, entropy, value = self.model.evaluate(
                    vec_obs[mb], point_cloud[mb], cont_acts[mb], disc_acts[mb]
                )

                ratio = (log_prob - old_log_prob[mb]).exp()
                adv = advantages[mb]
                surr1 = ratio * adv
                surr2 = ratio.clamp(1.0 - self.clip_eps, 1.0 + self.clip_eps) * adv
                policy_loss = -torch.min(surr1, surr2).mean()

                value_loss = (returns[mb] - value).pow(2).mean()
                entropy_loss = -entropy.mean()

                loss = policy_loss + self.vf_coef * value_loss + self.ent_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
                self.optimizer.step()

                total_pl += policy_loss.item()
                total_vl += value_loss.item()
                total_ent += entropy.mean().item()

        n_minibatches = (n + self.batch_size - 1) // self.batch_size
        steps = self.n_epochs * max(1, n_minibatches)
        return {
            "policy_loss": total_pl / steps,
            "value_loss": total_vl / steps,
            "entropy": total_ent / steps,
        }

## 5. 롤아웃 수집 (`train.py`)

환경을 직접 굴려 버퍼를 채우는 부분. ML-Agents 통신의 특성을 다룬다.

- 관측이 2개(벡터=rank2, 모기집합=rank3)라 **rank 로 구분**해서 분리
- **보상 타이밍(off-by-one) 주의**: 시점 t 에 정한 행동의 보상은 `env.step()` *이후*의
  `get_steps()` 에 도착한다. 그래서 행동을 보낸 뒤 다음 스텝 결과를 읽어 그 전이에 기록
- **종료 구분**: `terminal_steps.interrupted` 가 True=시간초과(truncated, V bootstrap),
  False=진짜 종료(성공/파손, bootstrap 0)

In [ ]:
def _split_batched_obs(obs_list, device):
    # decision_steps.obs -> (vec [B, vec_dim], points [B, N, 6])
    vec_np = next(o for o in obs_list if o.ndim == 2)      # 벡터는 rank-2
    pt_cands = [o for o in obs_list if o.ndim == 3]        # 모기 집합은 rank-3
    vec_t = torch.tensor(vec_np, dtype=torch.float32, device=device)
    if pt_cands:
        pt_t = torch.tensor(pt_cands[0], dtype=torch.float32, device=device)
    else:
        pt_t = torch.zeros(vec_t.shape[0], 0, 6, device=device)
    return vec_t, pt_t


def _terminal_value(model, term_step, device):
    # 시간초과된 에이전트의 terminal 관측에서 V 계산 (배치 차원 없음)
    obs = term_step.obs
    vec_np = next(o for o in obs if o.ndim == 1)
    pt_cands = [o for o in obs if o.ndim == 2]
    vec_t = torch.tensor(vec_np[None], dtype=torch.float32, device=device)
    if pt_cands:
        pt_t = torch.tensor(pt_cands[0][None], dtype=torch.float32, device=device)
    else:
        pt_t = torch.zeros(1, 0, 6, device=device)
    with torch.no_grad():
        return model.get_value(vec_t, pt_t)[0].item()


def collect_rollout(env, model, buf, device, behavior_name):
    model.eval()
    ep_rewards = []
    ep_reward = 0.0
    last_value = 0.0

    while buf.ptr < buf.buf_size:
        decision_steps, _ = env.get_steps(behavior_name)
        if len(decision_steps) == 0:
            env.step()           # 이번 프레임 결정 대기 에이전트 없음 -> 진행만
            continue

        vec_t, pt_t = _split_batched_obs(decision_steps.obs, device)
        with torch.no_grad():
            cont, disc_list, log_prob, value = model.get_action(vec_t, pt_t)

        # ActionTuple 구성 후 환경에 전달
        cont_np = cont.cpu().numpy()
        disc_np = torch.stack(disc_list, dim=-1).cpu().numpy().astype("int32")
        env.set_actions(behavior_name, ActionTuple(continuous=cont_np, discrete=disc_np))
        env.step()

        # 방금 행동의 결과(보상/종료)는 다음 get_steps 에서 도착
        next_dec, next_term = env.get_steps(behavior_name)

        for i in range(len(decision_steps)):
            agent_id = decision_steps.agent_id[i]
            if agent_id in next_term:
                ts = next_term[agent_id]
                reward = float(ts.reward)
                interrupted = bool(ts.interrupted)       # True=시간초과
                boundary = True
                bootstrap_value = _terminal_value(model, ts, device) if interrupted else 0.0
            elif agent_id in next_dec:
                reward = float(next_dec[agent_id].reward)
                boundary = False
                bootstrap_value = 0.0
            else:
                reward = 0.0
                boundary = False
                bootstrap_value = 0.0

            ep_reward += reward
            buf.add(
                vec_obs=vec_t[i].cpu(),
                point_cloud=pt_t[i].cpu(),
                cont_act=cont[i].cpu(),
                disc_act=torch.stack([d[i] for d in disc_list], dim=-1).cpu(),
                log_prob=log_prob[i].cpu(),
                reward=reward,
                value=value[i].cpu(),
                done=boundary,
                bootstrap_value=bootstrap_value,
            )

            if boundary:
                ep_rewards.append(ep_reward)
                ep_reward = 0.0
                last_value = 0.0
            else:
                last_value = value[i].item()

            if buf.ptr >= buf.buf_size:
                break

    model.train()
    return last_value, ep_rewards

## 6. 학습 루프 — 환경 연결 & 초기화

여기부터 실제로 실행하면 Unity 빌드가 뜬다. (`cfg["env_path"]` 가 None 이면 에디터에서 Play 를 눌러야 연결됨)

- `EngineConfigurationChannel` 로 `time_scale` 전달(배속)
- 환경에서 관측/행동 스펙을 읽어 차원 자동 추론 → ActorCritic 생성

In [ ]:
save_dir = Path(cfg["save_dir"])
save_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(save_dir / "tb"))

engine_channel = EngineConfigurationChannel()
env = UnityEnvironment(
    file_name=cfg["env_path"],
    seed=cfg["seed"],
    timeout_wait=120,
    no_graphics=cfg["no_graphics"],
    side_channels=[engine_channel],
)
engine_channel.set_configuration_parameters(time_scale=float(cfg["time_scale"]))
env.reset()

behavior_name = list(env.behavior_specs.keys())[0]
spec = env.behavior_specs[behavior_name]

# 관측 2개 중 벡터(1D)에서 차원, action_spec 에서 행동 차원 추론
vec_spec = next(s for s in spec.observation_specs if len(s.shape) == 1)
vec_dim = vec_spec.shape[0]
cont_dim = spec.action_spec.continuous_size
disc_sizes = list(spec.action_spec.discrete_branches)
print("behavior =", behavior_name, "| vec_dim =", vec_dim,
      "| cont =", cont_dim, "| disc =", disc_sizes)

In [ ]:
model = ActorCritic(
    vec_dim=vec_dim,
    pointnet_out=cfg["pointnet_out"],
    cont_dim=cont_dim,
    disc_sizes=disc_sizes,
).to(device)

ppo = PPO(
    model=model, lr=cfg["lr"], clip_eps=cfg["clip_eps"],
    vf_coef=cfg["vf_coef"], ent_coef=cfg["ent_coef"],
    max_grad_norm=cfg["max_grad_norm"], n_epochs=cfg["n_epochs"],
    batch_size=cfg["batch_size"],
)

buf = RolloutBuffer(
    buf_size=cfg["buf_size"], vec_dim=vec_dim, cont_dim=cont_dim,
    n_disc=len(disc_sizes), gamma=cfg["gamma"], lam=cfg["lam"], device=device,
)

## 7. 메인 루프

`max_iterations` 만큼 반복: 수집 → GAE → PPO 업데이트 → 로그/저장.
`log_interval` 마다 콘솔에 한 줄(`ep_reward` 가 nan 이면 그 iteration 에 에피소드가 한 번도 안 끝난 것).

In [ ]:
total_steps = 0
for iteration in range(cfg["max_iterations"]):
    buf.reset()
    last_value, ep_rewards = collect_rollout(env, model, buf, device, behavior_name)
    batch = buf.get(last_value)

    losses = ppo.update(batch)
    total_steps += buf.ptr

    writer.add_scalar("train/policy_loss", losses["policy_loss"], total_steps)
    writer.add_scalar("train/value_loss", losses["value_loss"], total_steps)
    writer.add_scalar("train/entropy", losses["entropy"], total_steps)
    if ep_rewards:
        writer.add_scalar("train/ep_reward_mean", sum(ep_rewards) / len(ep_rewards), total_steps)

    if (iteration + 1) % cfg["log_interval"] == 0:
        mean_r = sum(ep_rewards) / len(ep_rewards) if ep_rewards else float("nan")
        print(f"[{iteration+1:5d}] steps={total_steps:7d} | ep_reward={mean_r:.2f} | "
              f"policy={losses['policy_loss']:.4f} | value={losses['value_loss']:.4f} | "
              f"entropy={losses['entropy']:.4f}")

    if (iteration + 1) % cfg["save_interval"] == 0:
        path = save_dir / f"model_{iteration+1}.pt"
        torch.save({
            "model": model.state_dict(),
            "optimizer": ppo.optimizer.state_dict(),
            "iteration": iteration + 1,
            "total_steps": total_steps,
        }, path)
        print("  saved ->", path)

env.close()
writer.close()

## 정리

- **ActorCritic**: 벡터(MLP) + 모기집합(PointNet) → fusion → Actor(연속 Gaussian + 이산 Categorical) / Critic(V)
- **RolloutBuffer**: 경험 저장 + GAE(advantage/return) + advantage 정규화
- **PPO**: clip 목적함수로 정책/가치 업데이트
- **collect_rollout**: ML-Agents 통신(보상 off-by-one, truncated vs 종료 구분) 처리
- **루프**: 수집 → GAE → 업데이트 → 로그/저장

실제 학습은 `python scripts/run_train.py env_path=Builds/MainEnv/Chameleon_env.exe no_graphics=true time_scale=20`.
이 노트북은 그 내부를 펼쳐 본 것이며, `src/` 와 로직이 동일하다.